In [ ]:

"""
Notebook Scaffold: Object-Oriented Architecture (Declarations Only)

Naming Style Guide:
- Functions: snake_case
- Classes: PascalCase
- Methods: snake_case
- Parameters: explicit & typed
- Return types: predictable (np.ndarray, Dict[str, Any], etc.)
- Verb-first for actions: open_, read_, load_, infer_, generate_, export_
- Nouns for data objects: frame_bgr, frame_rgb, patch_rgb, detections, metrics
"""

from typing import Optional, Tuple, Iterator, List, Dict, Any
import numpy as np


import logging

# Configure root logger
logging.basicConfig(
    level=logging.DEBUG,  # Change to INFO for less verbosity
    format="[%(asctime)s] %(levelname)s:%(name)s: %(message)s",
    datefmt="%H:%M:%S"
)

# Component-specific loggers
camera_logger = logging.getLogger("camera")
preprocessor_logger = logging.getLogger("preprocessor")
detector_logger = logging.getLogger("detector")
metrics_logger = logging.getLogger("metrics")
attack_logger = logging.getLogger("attack")
printer_logger = logging.getLogger("printer")
orchestrator_logger = logging.getLogger("orchestrator")

# Reduce noise from external libraries
logging.getLogger("ultralytics").setLevel(logging.WARNING)
logging.getLogger("torch").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("PIL").setLevel(logging.WARNING)


# =========================================================
# 1) Camera — Live feed via OpenCV
# =========================================================
class Camera:
    """Physical camera device (OpenCV)."""

    def __init__(self,
                 index: int = 0,
                 width: Optional[int] = None,
                 height: Optional[int] = None,
                 backend: Optional[int] = None) -> None:
        """
        Configure camera parameters (not opening it yet).
        Idempotent: Yes.
        """

    def open(self) -> "cv2.VideoCapture":
        """
        Open the camera; return cv2.VideoCapture handle.
        Idempotent: Yes (fresh handle per call).
        """

    def read_frame(self, cap: "cv2.VideoCapture") -> "np.ndarray | None":
        """
        Read one BGR frame from capture; None if read fails.
        Idempotent: Yes (single read).
        """

    def frames(self,
               cap: "cv2.VideoCapture",
               resize: Optional[Tuple[int, int]] = None,
               flip_horizontal: bool = False) -> Iterator[Tuple[int, np.ndarray]]:
        """
        Generator yielding (frame_id, frame_bgr) with optional resize/flip.
        Idempotent per iteration.
        """

    def release(self, cap: "cv2.VideoCapture") -> None:
        """
        Release camera resource safely.
        Idempotent: Yes (safe to call multiple times).
        """


# =========================================================
# 2) Preprocessor — Convert, resize, normalize, tensorize
# =========================================================
class Preprocessor:
    """Transforms frames into model‑ready tensors."""

    def to_rgb(self, frame_bgr: np.ndarray) -> np.ndarray:
        """Convert BGR → RGB. Idempotent: Yes."""

    def resize(self, frame_rgb: np.ndarray, size: Tuple[int, int] = (640, 640)) -> np.ndarray:
        """Resize to model input resolution. Idempotent: Yes."""

    def normalize_01(self, frame_rgb: np.ndarray) -> np.ndarray:
        """Normalize pixel values to [0,1] float32. Idempotent: Yes."""

    def to_tensor(self, frame_rgb_norm: np.ndarray) -> "torch.Tensor":
        """Convert numpy → PyTorch tensor (expected layout). Idempotent: Yes."""


# =========================================================
# 3) Detector — YOLOv8 (Ultralytics, PyTorch)
# =========================================================
class Detector:
    """Object detection model (YOLOv8)."""

    def load_yolov8(self, weights: str = "yolov8n.pt", device: Optional[str] = None) -> Any:
        """
        Load Ultralytics YOLOv8 model.
        Output: model handle.
        Idempotent: Yes.
        """

    def infer(self,
              model: Any,
              image_bgr: np.ndarray,
              conf: float = 0.25,
              iou: float = 0.5,
              classes: Optional[List[int]] = [0]) -> Any:
        """
        Run one‑shot inference on a single BGR image.
        Output: Ultralytics Results object.
        Idempotent: Yes.
        """

    def extract_person(self, results: Any, conf_threshold: float = 0.25) -> List[Dict[str, Any]]:
        """
        Filter detections to 'person' (COCO class 0).
        Output: list of {bbox:(x1,y1,x2,y2), conf:float, class_id:int}.
        Idempotent: Yes.
        """

    def render(self, results: Any) -> np.ndarray:
        """
        Annotated image for display (.plot()).
        Output: frame_bgr with overlays.
        Idempotent: Yes.
        """


# =========================================================
# 4) Metrics — Baseline vs Attack comparison
# =========================================================
class Metrics:
    """Measurement utilities for frames and runs."""

    def frame(self, dets: List[Dict[str, Any]]) -> Dict[str, float]:
        """
        Per‑frame metrics: person_count, max_conf, avg_conf.
        Output: dict[str, float].
        Idempotent: Yes.
        """

    def compare(self, baseline: Dict[str, float], attack: Dict[str, float]) -> Dict[str, float]:
        """
        Compare baseline vs attack (Δcount, Δconfidence, ratios).
        Output: dict[str, float].
        Idempotent: Yes.
        """


# =========================================================
# 5) AttackGenerator — IBM ART (Adversarial Patch + EOT)
# =========================================================
class AttackGenerator:
    """Generates robust adversarial patch via ART."""

    def load_training_frames(self, paths: List[str]) -> List[np.ndarray]:
        """
        Load person images (COCO subset or custom).
        Output: list[np.ndarray].
        Idempotent: Yes.
        """

    def build_art_estimator(self, model: Any) -> Any:
        """
        Wrap model as ART estimator (adapter for OD flow).
        Output: ART estimator.
        Idempotent: Yes.
        """

    def generate_patch_eot(self,
                           estimator: Any,
                           frames: List[np.ndarray],
                           patch_shape: Tuple[int, int] = (280, 200),
                           rotation_max: float = 30.0,
                           scale_range: Tuple[float, float] = (0.6, 1.4),
                           brightness_range: Tuple[float, float] = (0.7, 1.3),
                           steps: int = 500) -> np.ndarray:
        """
        Optimize adversarial patch with EOT‑like transforms.
        Output: patch_rgb (np.ndarray).
        Idempotent: Yes (fixed seed/config).
        """

    def export_patch_cmyk(self,
                          patch_rgb: np.ndarray,
                          output_path: str,
                          dpi: int = 300,
                          icc_profile_path: Optional[str] = None) -> None:
        """
        Export high‑resolution CMYK patch for print workflow.
        Idempotent: Yes (overwrite policy controlled by caller).
        """


# =========================================================
# 6) Printer — Physicalization specs (T‑shirt)
# =========================================================
class Printer:
    """Prepares spec payload for patch printing on T‑shirt."""

    def build_specs(self,
                    width_cm: float = 20.0,
                    height_cm: float = 28.0,
                    margin_cm: float = 1.0) -> Dict[str, float]:
        """
        Return spec dict (size, margin, placement).
        Idempotent: Yes.
        """


# =========================================================
# 7) Orchestrator — Glue for single steps
# =========================================================
class Orchestrator:
    """Coordinates camera, preprocessor, detector, attack generation."""

    def init_runtime(self,
                     cam_index: int = 0,
                     cam_size: Optional[Tuple[int, int]] = None,
                     weights: str = "yolov8n.pt",
                     device: Optional[str] = None) -> Dict[str, Any]:
        """
        Initialize camera, model, and config.
        Output: {'cap':..., 'model':..., 'config':...}
        Idempotent: Yes.
        """

    def step_inference(self, runtime: Dict[str, Any]) -> Dict[str, Any]:
        """
        One inference step:
        read → preprocess → infer → extract → render → metrics.
        Output:
            {
              'frame_id': int,
              'raw_frame': np.ndarray,
              'annotated': np.ndarray,
              'detections': List[dict],
              'metrics': Dict[str, float]
            }
        Idempotent: Yes (for given frame).
        """

    def step_attack_generation(self,
                               runtime: Dict[str, Any],
                               train_paths: List[str],
                               export_path: str) -> Dict[str, Any]:
        """
        One attack generation step:
        load frames → ART estimator → patch → export.
        Output: {'patch_rgb': np.ndarray, 'export_path': str}
        Idempotent: Yes (fixed inputs/seed).
        """




# =========================================================
# Main Runner: Initialize runtime and run one inference step
# =========================================================
def main():
    # 1) Initialize orchestrator and runtime
    orchestrator = Orchestrator()
    runtime = orchestrator.init_runtime(
        cam_index=0,                # Camera index (0 = default webcam)
        cam_size=(640, 480),        # Optional resolution
        weights="yolov8n.pt",       # YOLOv8 weights
        device="cuda:0"             # or "cpu" if no GPU
    )

    # 2) Run one inference step
    result = orchestrator.step_inference(runtime)

    # 3) Print summary
    print(f"Frame ID: {result['frame_id']}")
    print(f"Detections: {result['detections']}")
    print(f"Metrics: {result['metrics']}")

    # 4) (Optional) Display annotated frame
    import cv2
    cv2.imshow("Annotated Frame", result['annotated'])
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# Execute main
if __name__ == "__main__":
    main()
